## Scenario: A software company wants to build its first AI assistant. 
### Tasks: Create a basic LLM-powered agent capable of handling user queries, maintaining context, and performing task execution workflows.

In [ ]:
import os
import time
from pathlib import Path

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

from langchain.agents import create_agent

from langgraph.checkpoint.memory import InMemorySaver

print("All libraries imported successfully!")

In [ ]:
PDF_PATH = "Viva Questions with Answers.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"PDF loaded successfully: {len(documents)} pages")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

In [ ]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1}")
    print(chunk.page_content)
    print("-" * 80)

In [ ]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model initialized successfully!")

In [ ]:
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(len(vectors), len(vectors[0]))

In [ ]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
results = retriever.invoke(
    "What are patient safety rights?"
)

for doc in results:
    print(doc.page_content)

In [ ]:
system_prompt = SystemMessage(
    content="""
You are an AI Hospital Assistant.
Answer using the hospital knowledge base.
Do not diagnose or recommend medical treatment.
If the answer is not found, say you don't know.
"""
)

memory = InMemorySaver()

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:llama3.2:3b",
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Agent created")

In [ ]:
def search_hospital(query):
    results = retriever.invoke(query)
    return "\n".join(doc.page_content for doc in results)

In [ ]:
from langchain_core.tools import tool

@tool
def search_hospital(query: str) -> str:
    """Search hospital documents."""
    results = retriever.invoke(query)
    return "\n".join(doc.page_content for doc in results)

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:llama3.2:3b",
    tools=[search_hospital],
    system_prompt=system_prompt,
    checkpointer=memory
)
print("Agent created")

In [ ]:
config = {
    "configurable": {
        "thread_id": "patient-001"
    }
}

In [ ]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What are patient safety rights?"
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

In [ ]:
import sys
print(sys.executable)